# ShqipAI Fine-Tuning: Phase 2 - TPU Training

**Purpose**: Fine-tune Gemma for educational tutoring

**Hardware**: TPU VM v5-8 (20 hours quota)

**Time**: ~12-15 hours

**IMPORTANT**: Set Accelerator to TPU VM v5-8 in Settings!

In [ ]:
# Install packages
!pip install -q torch_xla transformers datasets accelerate peft trl

import torch
import torch_xla.core.xla_model as xm

device = xm.xla_device()
print(f'TPU Device: {device}')

In [ ]:
# Hugging Face login
from huggingface_hub import login

HF_TOKEN = 'YOUR_TOKEN_HERE'  # Get from huggingface.co/settings/tokens
login(token=HF_TOKEN)

In [ ]:
# Load model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = 'google/gemma-2-9b-it'  # Change to gemma-4-12b-it when available

print(f'Loading {MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='auto', token=HF_TOKEN)
print(f'Model loaded: {model.num_parameters()/1e9:.1f}B parameters')

In [ ]:
# Load training data
from datasets import load_dataset
import glob

data_files = glob.glob('/kaggle/input/**/*.jsonl', recursive=True)
print(f'Found: {data_files}')

if data_files:
    dataset = load_dataset('json', data_files=data_files[0], split='train')
    print(f'Loaded {len(dataset)} examples')
else:
    print('ERROR: Upload your educational_data.jsonl file!')

In [ ]:
# Tokenize
def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, max_length=2048, padding='max_length')

print('Tokenizing...')
tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
print(f'Tokenized {len(tokenized)} examples')

In [ ]:
# Training config
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir='./shqipai-tutor',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    bf16=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=3,
    tpu_num_cores=8,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print('Ready to train!')

In [ ]:
# START TRAINING
print('='*60)
print('STARTING TRAINING - This takes 12-15 hours')
print('Watch the loss - it should decrease from ~2.0 to ~0.5')
print('DO NOT CLOSE THIS TAB!')
print('='*60)

trainer.train()

print('\nTRAINING COMPLETE!')

In [ ]:
# Save model
model.save_pretrained('./shqipai-tutor-final')
tokenizer.save_pretrained('./shqipai-tutor-final')
print('Model saved to ./shqipai-tutor-final')
!ls -la ./shqipai-tutor-final/

In [ ]:
# Test the model
prompt = 'Explain photosynthesis to a 10-year-old:'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.7)

print(tokenizer.decode(out[0], skip_special_tokens=True))